#### Tech Used:

* Python
* Pickle
* Dictionary-based cache
* Session tracking

###  Imports

In [18]:
import pickle
import faiss
import numpy as np

#### Load metadata:

In [19]:
with open("data/metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

print("Metadata loaded:", len(metadata))

Metadata loaded: 6


#### Load FAISS index:

In [20]:
index = faiss.read_index("data/faiss_index.bin")

print("FAISS index loaded!")

FAISS index loaded!


#### Load vectorizer:

In [23]:
with open("data/vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print("Vectorizer loaded!")

Vectorizer loaded!


### Recreate hybrid_search()

In [24]:
def hybrid_search(question, top_k=3):

    query_vector = vectorizer.transform([question]).toarray().astype("float32")

    distances, indices = index.search(query_vector, top_k)

    results = []

    for idx in indices[0]:

        results.append({

            "text": metadata[idx]["text"],

            "source": metadata[idx]["source"]

        })

    return results

#### Test

In [25]:
results = hybrid_search("What monitoring tools are used?")

results

[{'text': 'Cloud Migration Guide\n\nAWS is the primary cloud platform.\n\nKubernetes is used for deployment.\n\nGitHub Actions manages CI/CD.\n\nPrometheus and Grafana are used for monitoring.',
  'source': 'cloud_migration_guide.txt'},
 {'text': 'Employee Handbook\n\nEmployees receive 20 annual leave days.\n\nRemote work is allowed three days per week.\n\nWorking hours are 9 AM to 6 PM.\n\nHealth insurance benefits are provided.',
  'source': 'employee_handbook.txt'},
 {'text': 'Company Policies\n\nPasswords must be changed every 90 days.\n\nVPN access is mandatory.\n\nMulti-factor authentication is required.\n\nConfidential data should not be shared externally.',
  'source': 'company_policies.txt'}]

### Create Cache

In [26]:
cache = {}

print("Cache initialized successfully!")

Cache initialized successfully!


### Save to Cache Function

In [27]:
def save_to_cache(question, answer):

    cache[question] = answer

### Retrieve from Cache

In [28]:
def get_from_cache(question):

    return cache.get(question)

### Test Cache

In [29]:
sample_answer = {

    "answer": "Employees receive 20 annual leave days.",

    "sources": ["employee_handbook.txt"]

}

save_to_cache(

    "How many leave days are allowed?",

    sample_answer

)

print(

    get_from_cache(
        "How many leave days are allowed?"
    )

)

{'answer': 'Employees receive 20 annual leave days.', 'sources': ['employee_handbook.txt']}


### Create Session Memory

In [30]:
session_memory = {}

print("Session memory initialized!")

Session memory initialized!


### Add Conversation Function

In [31]:
def add_to_session(session_id, question, answer):

    if session_id not in session_memory:

        session_memory[session_id] = []

    session_memory[session_id].append({

        "question": question,

        "answer": answer

    })

### Get Session History

In [32]:
def get_session_history(session_id):

    return session_memory.get(session_id, [])

### Test Session Memory

In [33]:
add_to_session(

    "user_1",

    "How many leave days are allowed?",

    "Employees receive 20 annual leave days."

)

print(

    get_session_history("user_1")

)

[{'question': 'How many leave days are allowed?', 'answer': 'Employees receive 20 annual leave days.'}]


 ### Cached Search Function

In [34]:
def cached_search(question):

    cached_result = get_from_cache(question)

    if cached_result:

        print("Retrieved from cache")

        return cached_result

    results = hybrid_search(question)

    answer = {

        "answer": results[0]["text"],

        "sources": [results[0]["source"]]

    }

    save_to_cache(question, answer)

    print("Saved to cache")

    return answer

### First Search

In [35]:
response = cached_search(

    "What monitoring tools are used?"

)

print(response)

Saved to cache
{'answer': 'Cloud Migration Guide\n\nAWS is the primary cloud platform.\n\nKubernetes is used for deployment.\n\nGitHub Actions manages CI/CD.\n\nPrometheus and Grafana are used for monitoring.', 'sources': ['cloud_migration_guide.txt']}


### Second Search

In [36]:
response = cached_search(

    "What monitoring tools are used?"

)

print(response)

Retrieved from cache
{'answer': 'Cloud Migration Guide\n\nAWS is the primary cloud platform.\n\nKubernetes is used for deployment.\n\nGitHub Actions manages CI/CD.\n\nPrometheus and Grafana are used for monitoring.', 'sources': ['cloud_migration_guide.txt']}


### Save Session Memory

In [37]:
with open("data/session_memory.pkl", "wb") as f:

    pickle.dump(session_memory, f)

print("Session memory saved successfully!")

Session memory saved successfully!


### Load Session Memory

In [38]:
with open("data/session_memory.pkl", "rb") as f:

    loaded_memory = pickle.load(f)

print(loaded_memory)

{'user_1': [{'question': 'How many leave days are allowed?', 'answer': 'Employees receive 20 annual leave days.'}]}


### Verify Files

In [39]:
import os

print(os.listdir("data"))

['bm25.pkl', 'chunks.pkl', 'csv', 'documents.pkl', 'embeddings.npy', 'faiss_index.bin', 'metadata.pkl', 'pdfs', 'session_memory.pkl', 'text_docs', 'vectorizer.pkl']
